[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-10b-serving-and-testing-a-model.ipynb)

# Advanced Discussion 10 (continued) — a validated prediction API and pytest for ML code

A FastAPI prediction endpoint with input validation, online-versus-batch timings, and a nine-test pytest suite checked against six injected bugs.

**Optional advanced-discussion lab** for Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Runs in about two minutes on CPU. Timings depend on your machine; the ratios are what matter.

In [ ]:
%pip install -q torch scikit-learn fastapi httpx pytest pydantic

In [ ]:
import os, sys, time, json, subprocess, tempfile, textwrap
import numpy as np, torch

WORK = tempfile.mkdtemp(prefix="dl_serving_")

## 1. the code under test: a small classifier package with switchable bugs (BUG env var) so we can see which test catches which mistake

In [ ]:
open(os.path.join(WORK, "mymodel.py"), "w").write(textwrap.dedent('''
    import os, numpy as np, torch
    from sklearn.datasets import load_digits
    BUG = os.environ.get("BUG", "")

    def load_data(seed=0):
        X, y = load_digits(return_X_y=True); X = (X / 16.0).astype("float32")
        rng = np.random.RandomState(seed); idx = rng.permutation(len(X)); cut = int(0.7 * len(X))
        tr, te = idx[:cut], idx[cut:]
        if BUG == "leak": te = np.concatenate([te, tr[:200]])                        # test set accidentally contains training rows
        Xtr, ytr = X[tr], y[tr]
        if BUG == "labels_shifted": ytr = (ytr + 1) % 10                              # off-by-one in the TRAINING labels only
        if BUG == "misaligned": ytr = ytr[rng.permutation(len(ytr))]                  # X and y shuffled independently
        return (torch.tensor(Xtr), torch.tensor(ytr)), (torch.tensor(X[te]), torch.tensor(y[te])), (tr, te)

    def build_model(seed=0):
        if BUG != "unseeded": torch.manual_seed(seed)
        layers = [torch.nn.Linear(64, 64), torch.nn.BatchNorm1d(64), torch.nn.ReLU(), torch.nn.Dropout(0.3), torch.nn.Linear(64, 10)]
        return torch.nn.Sequential(*layers)

    def train(model, X, y, steps=200, lr=1e-2):
        opt = torch.optim.Adam(model.parameters(), lr); model.train(); losses = []
        for _ in range(steps):
            opt.zero_grad(); loss = torch.nn.functional.cross_entropy(model(X), y); loss.backward(); opt.step(); losses.append(loss.item())
        return losses

    def predict_proba(model, X):
        if BUG != "no_eval": model.eval()                                             # forgetting this leaves dropout and batch-norm in training mode
        with torch.no_grad():
            logits = model(X)
            return logits if BUG == "no_softmax" else torch.softmax(logits, 1)        # a service must return probabilities
'''))
open(os.path.join(WORK, "test_model.py"), "w").write(textwrap.dedent('''
    import numpy as np, torch, pytest
    from mymodel import load_data, build_model, train, predict_proba

    @pytest.fixture(scope="module")
    def data(): return load_data()

    def test_output_shape(data):
        (X, y), _, _ = data; assert predict_proba(build_model(), X[:5]).shape == (5, 10)

    def test_probabilities_sum_to_one(data):
        (X, y), _, _ = data; p = predict_proba(build_model(), X[:20]); assert torch.allclose(p.sum(1), torch.ones(20), atol=1e-5)

    def test_same_seed_same_weights(data):
        (X, y), _, _ = data; a, b = build_model(0), build_model(0)
        assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))

    def test_inference_is_deterministic(data):
        (X, y), _, _ = data; m = build_model(); train(m, X, y, steps=20)
        assert torch.equal(predict_proba(m, X[:30]), predict_proba(m, X[:30]))

    def test_prediction_does_not_depend_on_batch_mates(data):
        (X, y), _, _ = data; m = build_model(); train(m, X, y, steps=20)
        alone = predict_proba(m, X[:1]); in_batch = predict_proba(m, X[:64])[:1]; assert torch.allclose(alone, in_batch, atol=1e-5)

    def test_can_overfit_one_small_batch(data):
        (X, y), _, _ = data; m = build_model(); losses = train(m, X[:16], y[:16], steps=300, lr=2e-2); assert losses[-1] < 0.1

    def test_train_and_test_rows_are_disjoint(data):
        _, _, (tr, te) = data; assert len(set(tr.tolist()) & set(te.tolist())) == 0

    def test_heldout_accuracy_beats_a_floor(data):
        (X, y), (Xt, yt), _ = data; m = build_model(); train(m, X, y, steps=200)
        assert (predict_proba(m, Xt).argmax(1) == yt).float().mean() > 0.90

    def test_save_load_roundtrip(data, tmp_path):
        (X, y), _, _ = data; m = build_model(); train(m, X, y, steps=20); p = predict_proba(m, X[:10])
        torch.save(m.state_dict(), tmp_path / "w.pt"); m2 = build_model(1); m2.load_state_dict(torch.load(tmp_path / "w.pt")); assert torch.allclose(p, predict_proba(m2, X[:10]), atol=1e-6)
'''))

def run_tests(bug):
    env = dict(os.environ, BUG=bug, PYTHONPATH=WORK)
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--tb=no", "-p", "no:cacheprovider", "-rf", os.path.join(WORK, "test_model.py")], capture_output=True, text=True, cwd=WORK, env=env)
    failed = sorted({l.split("::")[1].split(" ")[0] for l in r.stdout.splitlines() if l.startswith("FAILED")})
    summary = [l for l in r.stdout.splitlines() if " passed" in l or " failed" in l][-1:]
    return failed, summary

print("---- pytest on the correct code")
failed, summary = run_tests(""); print("failed tests:", failed or "none", "|", summary)

## 2. inject one bug at a time: which test notices?

In [ ]:
bugs = [("no_eval", "forgot model.eval() at inference"), ("no_softmax", "returned logits instead of probabilities"), ("unseeded", "model initialised without a seed"),
        ("leak", "test set contains training rows"), ("labels_shifted", "training labels off by one"), ("misaligned", "features and labels shuffled independently")]
print("\nbug injected                                   tests that fail")
for bug, desc in bugs:
    failed, _ = run_tests(bug); print("%-46s %s" % (desc, ", ".join(f.replace("test_", "") for f in failed) or "NONE (undetected)"))

## 3. serve it: a validated prediction endpoint, exercised in-process

In [ ]:
os.environ["BUG"] = ""; sys.path.insert(0, WORK)
import importlib, mymodel; importlib.reload(mymodel)
from mymodel import load_data, build_model, train, predict_proba
(Xtr, ytr), (Xte, yte), _ = load_data(); model = build_model(); train(model, Xtr, ytr, steps=200); model.eval()
print("\nheld-out accuracy of the model we will serve: %.3f" % (predict_proba(model, Xte).argmax(1) == yte).float().mean())
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator
MODEL_VERSION = "digits-mlp-2026.10.1"
class One(BaseModel):
    pixels: list[float] = Field(min_length=64, max_length=64)
    @field_validator("pixels")
    @classmethod
    def in_range(cls, v):
        if min(v) < 0 or max(v) > 1: raise ValueError("pixel values must lie in [0, 1]")
        return v
class Many(BaseModel): items: list[One] = Field(min_length=1, max_length=1000)
app = FastAPI()
@app.get("/health")
def health(): return {"status": "ok", "model_version": MODEL_VERSION}
@app.post("/predict")
def predict(req: One):
    p = predict_proba(model, torch.tensor([req.pixels], dtype=torch.float32))[0]; return {"label": int(p.argmax()), "confidence": round(float(p.max()), 4), "model_version": MODEL_VERSION}
@app.post("/predict_batch")
def predict_batch(req: Many):
    p = predict_proba(model, torch.tensor([i.pixels for i in req.items], dtype=torch.float32)); return {"labels": p.argmax(1).tolist(), "model_version": MODEL_VERSION}
c = TestClient(app); x = Xte[0].tolist()
print("GET /health ->", c.get("/health").json())
r = c.post("/predict", json={"pixels": x}); print("POST /predict ->", r.status_code, r.json(), "(true label %d)" % int(yte[0]))
r = c.post("/predict", json={"pixels": x[:63]}); print("63 values ->", r.status_code, r.json()["detail"][0]["msg"])
r = c.post("/predict", json={"pixels": [2.0] + x[1:]}); print("value 2.0 ->", r.status_code, r.json()["detail"][0]["msg"])
r = c.post("/predict", json={"pixels": ["a"] * 64}); print("text values ->", r.status_code, r.json()["detail"][0]["msg"])

## 4. online, micro-batched, batch: the same model, three ways of calling it (timings are machine dependent; ratios are what matter)

In [ ]:
N = 300; items = [{"pixels": Xte[i].tolist()} for i in range(N)]
t0 = time.perf_counter()
for it in items: c.post("/predict", json=it)
t_single = time.perf_counter() - t0
t0 = time.perf_counter(); c.post("/predict_batch", json={"items": items}); t_batch = time.perf_counter() - t0
t0 = time.perf_counter()
for i in range(N): predict_proba(model, Xte[i:i + 1])
t_direct1 = time.perf_counter() - t0
t0 = time.perf_counter(); predict_proba(model, Xte[:N]); t_directN = time.perf_counter() - t0
print("\n%d predictions, this machine:" % N)
print("  one HTTP request each        : %7.1f ms per prediction (%6.0f per second)" % (1000 * t_single / N, N / t_single))
print("  one HTTP request, %d in a batch: %7.2f ms per prediction (%6.0f per second)  -> %.0fx" % (N, 1000 * t_batch / N, N / t_batch, t_single / t_batch))
print("  model called one row at a time: %7.3f ms per prediction" % (1000 * t_direct1 / N))
print("  model called on all %d rows   : %7.4f ms per prediction  -> %.0fx faster than row by row" % (N, 1000 * t_directN / N, t_direct1 / t_directN))
print("  overhead of the web layer per single request: about %.1f ms on top of %.3f ms of model time" % (1000 * (t_single - t_direct1) / N, 1000 * t_direct1 / N))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.